# Begleitnotizbuch zu Kapitel 6
**Build Your First LLM — Kapitel 6: NumPy & PyTorch Überlebensführer**

Führe diese Zellen von oben nach unten aus, um die Tensor-Grundlagen, Maskierung, Broadcasting und eine kleine Trainingsschleife zu sehen.

In [ ]:
# ===== IMPORTE =====
# torch: Die PyTorch-Bibliothek für Tensor-Operationen (denke: mehrdimensionale Arrays)
# numpy: Die klassische Bibliothek für numerisches Rechnen (PyTorch ist davon inspiriert)
# torch.nn: Bausteine für neuronale Netze (Schichten, etc.)
import torch, numpy as np
import torch.nn as nn

print('Torch-Version:', torch.__version__)
print('CUDA verfügbar:', torch.cuda.is_available())
# CUDA = GPU-Computing. Wenn True, können wir GPU-Beschleunigung nutzen (10-100× schneller für Deep Learning)

## Was ist ein Tensor?

Ein **Tensor** ist ein mehrdimensionales Array von Zahlen:
- **1D-Tensor** = eine Liste `[1, 2, 3]` (wie eine Zeile in einer Tabelle)
- **2D-Tensor** = eine Tabelle/Matrix (Zeilen und Spalten)
- **3D-Tensor** = ein Stapel von Tabellen (wie mehrere Blätter in Excel)

**Warum PyTorch statt NumPy?**
Beide verarbeiten mehrdimensionale Arrays, aber PyTorch fügt hinzu:
1. **GPU-Unterstützung** — Daten auf GPU verschieben für 10-100× Geschwindigkeitsgewinn
2. **Automatische Gradienten** — berechnet Ableitungen für das Training neuronaler Netze
3. **Neuronale Netz-Schichten** — vorgefertigte Bausteine

## Tensoren erstellen
Aus Python/NumPy-Daten und mit gängigen Füllregeln.

In [ ]:
# ===== Tensoren erstellen =====

# Aus Python-Daten
data = torch.tensor([[1, 2, 3], [4, 5, 6]])  # 2×3 Tensor

# Füllregeln (erstelle Tensoren mit spezifischen Werten gefüllt)
zeros = torch.zeros(3, 4)     # 3×4 Tensor mit Nullen
ones = torch.ones(2, 3, 4)    # 2×3×4 Tensor mit Einsen
uniform = torch.rand(3, 4)    # Zufallswerte im Bereich [0, 1)
normal = torch.randn(3, 4)    # Zufallswerte aus Normalverteilung (Mittelwert=0, Standardabweichung=1)
integers = torch.randint(0, 10, (3, 4))  # Zufällige Ganzzahlen im Bereich [0, 10)

# Bereichssequenzen (wie Pythons range)
sequence = torch.arange(0, 10, 2)  # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)  # 5 gleichmäßig verteilte Punkte von 0 bis 1

# Datentypen (dtype) — Präzision kontrollieren
float_tensor = torch.tensor([1.0, 2.0], dtype=torch.float32)  # 32-Bit Fließkommazahlen (Standard)
int_tensor = torch.tensor([1, 2], dtype=torch.long)           # 64-Bit Ganzzahlen (für Indizes)

# NumPy ↔ PyTorch (sie teilen sich den Speicher — Änderungen an einem betreffen das andere!)
np_data = np.array([1, 2, 3], dtype=np.float32)
torch_from_np = torch.from_numpy(np_data)  # teilt sich Speicher mit np_data
back_to_np = torch_from_np.numpy()         # zurück zu NumPy

# Geräteplatzierung — CPU oder GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_tensor = torch.randn(3, 4, device=device, dtype=torch.float32)

print('zeros Form:', zeros.shape)
print('gpu_tensor Gerät:', gpu_tensor.device)

## Reproduzierbarkeit: Seeds setzen
Kritisch für Debugging und Vergleich von Experimenten

In [ ]:
# Zufallszustand für Reproduzierbarkeit fixieren
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Diese sind jetzt bei jedem Durchlauf identisch
a = torch.randn(3, 4)
b = torch.randn(3, 4)

# Ohne Seed — Ergebnisse ändern sich bei jedem Durchlauf
# Aber mit Seed sind sie reproduzierbar
torch.manual_seed(42)
x1 = torch.randn(2, 3)
torch.manual_seed(42)
x2 = torch.randn(2, 3)
print('Mit Seed stimmen Tensoren überein:', torch.allclose(x1, x2))

# Warum das wichtig ist: Reproduzierbare Experimente für Debugging und Forschung

## Von Listen zu Tensoren: Dimensionen aufbauen
Verbinde die einfachen Listen aus Kapitel 5 mit mehrdimensionalen Tensoren

In [ ]:
# Schritt 1: 1D-Tensoren (Kapitel 5 Rückblick)
# Aus Kapitel 5: Token-IDs vom Tokenizer
token_ids = [2, 3, 4, 6]
tokens_1d = torch.tensor(token_ids)
print(f"1D Form: {tokens_1d.shape}")  # torch.Size([4])
print(f"Daten: {tokens_1d}")

In [ ]:
# Schritt 2: Batching (1D → 2D)
# Mehrere Sätze gleichzeitig verarbeiten
batch = torch.tensor([
    [2, 3, 4, 6],    # Satz 1
    [5, 7, 8, 9]     # Satz 2
])
print(f"2D-Batch Form: {batch.shape}")  # torch.Size([2, 4])
print("Erster Satz:", batch[0])
print("Zweites Token des ersten Satzes:", batch[0, 1])

In [ ]:
# Schritt 3: Einbettungen (2D → 3D)
# 768 Zahlen pro Token hinzufügen (GPT-2 Stil)
embeddings_3d = torch.randn(2, 4, 768)
print(f"3D-Einbettungen Form: {embeddings_3d.shape}")  # torch.Size([2, 4, 768])

# Durch Dimensionen navigieren: Batch → Token → Merkmale
print("Erste Satz-Einbettungen:", embeddings_3d[0].shape)      # (4, 768)
print("Erstes Token des ersten Satzes:", embeddings_3d[0, 0].shape)  # (768,)

In [ ]:
# Schritt 4: Aufmerksamkeitsköpfe Vorschau (3D → 4D)
# Multi-Head-Attention fügt eine weitere Dimension hinzu (keine Sorge um Details)
attention_4d = torch.randn(2, 8, 4, 4)  # (Batch, Köpfe, Sequenz, Sequenz)
print(f"4D-Attention Form: {attention_4d.shape}")
print("Form-Interpretation: (Batch, Köpfe, Sequenzlänge, Sequenzlänge)")

## Umformen, Quetschen, Permutieren
Form-Gymnastik, die du ständig verwenden wirst.

In [ ]:
# ===== Umformoperationen =====
# Umformen = Daten reorganisieren ohne Werte zu ändern (wie Kisten in einem Lager umordnen)

x = torch.arange(12)               # 1D-Tensor erstellen [0,1,2,...,11], Form (12,)
print(f'Start: {x.shape}')

# view() — umformen, erfordert aber "zusammenhängenden" Speicher (Daten sequenziell angeordnet)
x = x.view(3, 4)                   # Umformen zu (3, 4): 3 Zeilen × 4 Spalten
print(f'Nach view(3,4): {x.shape}')

# reshape() — wie view(), behandelt aber nicht-zusammenhängende Tensoren (sicherer, nutze dies bei Unsicherheit)
x = x.reshape(2, 2, 3)             # Umformen zu (2, 2, 3)
print(f'Nach reshape(2,2,3): {x.shape}')

# -1 bedeutet "finde diese Dimension für mich heraus"
x = x.view(-1, 3)                  # -1 wird zu 4 (12 Gesamtelemente ÷ 3 = 4)
print(f'Nach view(-1,3): {x.shape}')

# unsqueeze/squeeze — Dimensionen der Größe 1 hinzufügen oder entfernen
y = torch.randn(3, 4)
y = y.unsqueeze(0)                 # (1, 3, 4) — Batch-Dimension an Position 0 hinzufügen
y = y.unsqueeze(-1)                # (1, 3, 4, 1) — Dimension am Ende hinzufügen
y = y.squeeze()                    # ALLE Dimensionen der Größe 1 entfernen → (3, 4)
print(f'Nach squeeze: {y.shape}')

# permute — Dimensionen neu anordnen (wie Transponieren, aber für beliebige Anzahl von Dimensionen)
z = torch.randn(2, 3, 4)           # (Batch, Sequenz, Merkmale)
z = z.permute(0, 2, 1)             # (Batch, Merkmale, Sequenz) — letzten zwei Dimensionen tauschen
print(f'Nach permute: {z.shape}')

# flatten — Dimensionen zusammenklappen
t = torch.randn(2, 3, 4)
t_flat = t.flatten()               # (24,) — alle Dimensionen zusammengeklappt
t_flat_features = t.flatten(1)     # (2, 12) — abflachen ab Dimension 1
print(f'Vollständig abgeflacht: {t_flat.shape}')
print(f'Merkmale abflachen: {t_flat_features.shape}')

## Indizierung und Slicing
Batches, Tokens auswählen und boolesche Masken verwenden.

In [ ]:
# ===== Indizierung und Slicing =====
# Mehrdimensionale Daten navigieren wie du Ordner navigieren würdest: Batch → Token → Merkmale

# Einen 4D-Tensor erstellen, der Attention simuliert: (Batch, Köpfe, Sequenz, Sequenz)
x = torch.randn(2, 4, 6, 6)  # 2 Batches, 4 Köpfe, 6 Tokens, 6 Tokens

# Grundlegende Indizierung
first_batch = x[0]              # Form: (4, 6, 6) - erster Batch, alle Köpfe
first_head = x[0, 0]            # Form: (6, 6) - erster Batch, erster Kopf
single_value = x[0, 0, 0, 0]    # Form: () - eine einzelne Zahl

# Slicing mit Doppelpunkten
first_two_batches = x[:2]       # Form: (2, 4, 6, 6) - erste 2 Batches
all_but_last_token = x[:, :, :-1, :]  # Form: (2, 4, 5, 6) - letztes Token entfernen
every_other_head = x[:, ::2]    # Form: (2, 2, 6, 6) - Köpfe 0 und 2

# Negative Indizes zählen vom Ende
last_token = x[:, :, -1, :]     # Form: (2, 4, 6) - letztes Token in jeder Sequenz

# Boolesche Maskierung (nach Bedingung filtern)
mask = torch.tensor([True, False, True, False])
filtered_heads = x[0, mask]     # Form: (2, 6, 6) - nur Köpfe 0 und 2

print(f'Originalform: {x.shape}')
print(f'Erste Batch-Form: {first_batch.shape}')
print(f'Alle außer letztes Token: {all_but_last_token.shape}')
print(f'Gefilterte Köpfe: {filtered_heads.shape}')

## Attention: Das Herz der Transformer

**Das große Bild:** Attention berechnet einen gewichteten Durchschnitt aller Wörter, wobei die Gewichte aus Relevanz-Scores stammen. Wie beim Lesen von "Die Katze saß auf der Matte" — bei der Verarbeitung von "saß" schaust du zurück zu "Katze" (wer saß?) und "Matte" (wo saß?).

Wir bauen dies in 3 Schritten:
1. Grundlegende Mathematik (4 Operationen)
2. Kausale Maskierung hinzufügen (zukünftiges Spicken verhindern)
3. Produktionsabkürzung (PyTorch macht alles)

In [ ]:
import torch
import torch.nn.functional as F

# ===== Die grundlegende Mathematik von Attention =====
# Attention: Query (was suche ich?), Key (was habe ich?), Value (was soll zurückgegeben werden?)

# Einbettungen für 5 Tokens in Batch von 2 simulieren
batch, seq_len, d_head = 2, 5, 64
Q = torch.randn(batch, seq_len, d_head)  # (2, 5, 64)
K = torch.randn(batch, seq_len, d_head)
V = torch.randn(batch, seq_len, d_head)

# Schritt 1: Scores berechnen (wie sehr passt jedes Token zu jedem anderen?)
# Der @-Operator ist Matrixmultiplikation (dasselbe wie torch.matmul)
# K.transpose(-2, -1) tauscht die letzten beiden Dimensionen: (2, 5, 64) → (2, 64, 5)
# Ergebnis: (2, 5, 64) @ (2, 64, 5) → (2, 5, 5) — ein 5×5 Gitter von Scores pro Batch
scores = Q @ K.transpose(-2, -1)
print(f'Scores Form: {scores.shape}')  # (2, 5, 5)

# Schritt 2: Skalieren (Softmax-Sättigung verhindern)
# Ohne Skalierung, große Punktprodukte → Softmax gibt ~1 für max, ~0 für andere
scores = scores / (d_head ** 0.5)  # dividieren durch sqrt(64) = 8

# Schritt 3: Softmax (Scores in Wahrscheinlichkeiten umwandeln)
# dim=-1 bedeutet "entlang der letzten Dimension" (über Keys)
attn_weights = torch.softmax(scores, dim=-1)  # jede Zeile summiert sich zu 1

# Schritt 4: Gewichtete Summe (Werte mit Wahrscheinlichkeiten mischen)
output = attn_weights @ V  # (2, 5, 64)

print(f'Ausgabeform: {output.shape}')  # gleich wie Eingabe: (2, 5, 64)
print(f'Aufmerksamkeitsgewichte summieren zu 1: {attn_weights[0, 0].sum():.4f}')
print(f'\nToken 2 Aufmerksamkeitsgewichte: {attn_weights[0, 2]}')
print('(zeigt, wie sehr Token 2 auf jedes der 5 Tokens achtet)')

## Token- und Positionseinbettungen

Jedes LLM beginnt damit, Token-IDs in dichte Vektoren umzuwandeln.

**Das Problem:** Neuronale Netze können keinen rohen Text wie "Katze" verarbeiten. Token-IDs (wie `5` für "Katze") sind willkürlich — ID 5 ist nicht "näher" an 6 als an 500.

**Die Lösung:** Jede Token-ID auf einen gelernten Vektor abbilden (768 Zahlen für GPT-2). Ähnliche Wörter lernen durch Training ähnliche Vektoren.

In [ ]:
# ===== Token- und Positionseinbettungen =====
# GPT-2 Dimensionen
vocab_size, d_model, max_seq_len = 50257, 768, 1024

# Token-Einbettungen: eine Nachschlagetabelle (wie ein Dictionary: token_id → Vektor)
# nn.Embedding erstellt eine Tabelle mit vocab_size Zeilen und d_model Spalten
# Jede Zeile ist ein 768-dimensionaler Vektor, der ein Token repräsentiert
token_embedding = nn.Embedding(vocab_size, d_model)

# Einige zufällige Token-IDs erstellen (stelle vor, diese kämen von einem Tokenizer)
token_ids = torch.randint(0, vocab_size, (2, 5))  # 2 Sätze, 5 Tokens jeweils

# Einbettungen nachschlagen (nur Indizierung in die Tabelle!)
token_vectors = token_embedding(token_ids)
print(f'Token-Einbettungen: {token_vectors.shape}')  # (2, 5, 768)

# Positionseinbettungen: wo in der Sequenz (Token 0, Token 1, etc.)
# Gleiche Idee: eine Nachschlagetabelle, wo Zeile i "Position i" repräsentiert
pos_embedding = nn.Embedding(max_seq_len, d_model)
position_ids = torch.arange(5).unsqueeze(0).expand(2, -1)  # [[0,1,2,3,4], [0,1,2,3,4]]
pos_vectors = pos_embedding(position_ids)
print(f'Positionseinbettungen: {pos_vectors.shape}')  # (2, 5, 768)

# Kombinieren: elementweise Addition (gleiche Position, gleiche Form!)
# Warum addieren statt verketten? Addition hält Dimension bei 768 (nicht 1536)
# Das Modell lernt, SOWOHL Bedeutung ALS AUCH Position im gleichen Vektor zu kodieren
input_embeddings = token_vectors + pos_vectors
print(f'Kombiniert: {input_embeddings.shape}')  # (2, 5, 768)

# Parameterzählung (wie viele Zahlen zu lernen?)
token_params = vocab_size * d_model   # 50.257 Tokens × 768 Dimensionen
pos_params = max_seq_len * d_model    # 1.024 Positionen × 768 Dimensionen
print(f'Token-Parameter: {token_params:,}')      # 38.597.376
print(f'Positions-Parameter: {pos_params:,}')     # 786.432
print(f'Gesamt: {token_params + pos_params:,}')  # 39.383.808

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Einfaches 2-Schichten-MLP für Toy-Klassifikation (in sich geschlossen)
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

# Das 5-Schritte-Trainingsrezept:
# 1. Vorwärtsdurchlauf -> Vorhersagen erhalten
# 2. Verlust berechnen -> Fehler messen
# 3. Rückwärtsdurchlauf -> Gradienten berechnen
# 4. Gradienten clippen -> Explosionen verhindern
# 5. Gewichte aktualisieren -> Parameter anpassen

# Setup: Modell, Optimierer, Verlustfunktion, Fake-Daten
model = SimpleMLP(16, 32, 2)  # Eingabe 16-dimensional, versteckt 32-dimensional, Ausgabe 2 Klassen
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # adaptive Lernrate
loss_fn = nn.CrossEntropyLoss()  # für Klassifikation

# Fake-Trainingsdaten (batch_size=64)
inputs = torch.randn(64, 16)          # 64 Beispiele, 16 Merkmale jeweils
labels = torch.randint(0, 2, (64,))   # 64 Labels (Klasse 0 oder 1)

# Trainingsschleife (3 Epochen)
for epoch in range(3):
    # ===== Trainingsphase =====
    model.train()  # Dropout/Batch-Normalisierung aktivieren (falls vorhanden)

    # Schritt 1: Alte Gradienten auf Null setzen (sie akkumulieren standardmäßig!)
    optimizer.zero_grad(set_to_none=True)  # set_to_none spart Speicher

    # Schritt 2: Vorwärtsdurchlauf
    logits = model(inputs)  # Vorhersagen erhalten (rohe Scores)

    # Schritt 3: Verlust berechnen
    loss = loss_fn(logits, labels)  # Wie falsch liegen wir?

    # Schritt 4: Rückwärtsdurchlauf (Gradienten berechnen)
    loss.backward()  # .grad für jeden Parameter füllen

    # Schritt 5: Gradientenclipping (verhindert explodierende Gradienten)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Schritt 6: Gewichte aktualisieren
    optimizer.step()  # Parameter mit Gradienten anpassen

    print(f"Epoche {epoch}: Verlust={loss.item():.4f}")

# ===== Evaluierungsphase =====
model.eval()  # Dropout/Batch-Normalisierung deaktivieren
with torch.no_grad():  # Gradienten nicht verfolgen (spart Speicher)
    preds = model(inputs).argmax(dim=-1)  # Klassenvorhersagen erhalten
    accuracy = (preds == labels).float().mean()  # Anteil korrekt
    print(f"Genauigkeit: {accuracy:.2f}")

print()
print("✅ Wichtige Punkte:")
print("  - .backward() füllt das .grad jedes Parameters")
print("  - Optimierer nutzt Gradienten zur Gewichtsanpassung")
print("  - Gradientenclipping verhindert Verlustspitzen (kritisch für LLMs!)")
print("  - .eval() und no_grad() sparen Speicher während der Evaluierung")

In [ ]:
# Vollständiges Einbettungsmodul
class GPT2Embeddings(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        
        token_emb = self.token_embedding(token_ids)
        pos_emb = self.pos_embedding(position_ids)
        
        embeddings = token_emb + pos_emb
        embeddings = self.dropout(embeddings)
        return embeddings

# Testen
embed_layer = GPT2Embeddings(50257, 1024, 768)
token_ids = torch.randint(0, 50257, (2, 10))
output = embed_layer(token_ids)
print(f"Einbettungsausgabe: {output.shape}")  # (2, 10, 768)

# Gesamtparameter
total_params = sum(p.numel() for p in embed_layer.parameters())
print(f"Gesamte Einbettungsparameter: {total_params:,}")  # 39.383.808

## Die Trainingsschleife

**Schlüsselkonzepte:**
- `nn.Module`: Basisklasse für neuronale Netz-Schichten. Dein Modell erbt davon.
- `super().__init__()`: Ruft die Initialisierung der Elternklasse auf (erforderlicher Boilerplate)
- `nn.Linear(in, out)`: Eine Matrixmultiplikationsschicht (in×out Gewichtsmatrix)
- `nn.ReLU()`: Aktivierungsfunktion — behält positive Werte, setzt negative auf Null
- `CrossEntropyLoss`: Misst, wie falsch Klassifikationsvorhersagen sind
- `optimizer.zero_grad()`: Löscht alte Gradienten (sie akkumulieren standardmäßig!)
- `.backward()`: Berechnet Gradienten durch automatische Differentiation
- `.step()`: Aktualisiert Gewichte mit den berechneten Gradienten

In [ ]:
# Schritt 2: Kausale Maskierung hinzufügen (Schummeln verhindern)

# Problem: Wenn Token 2 Tokens 3 und 4 sehen kann, kann es beim Training schummeln!
# Lösung: Zukünftige Positionen blockieren, indem ihre Scores auf -inf gesetzt werden

# Wie die Maske aussieht (False = erlauben, True = blockieren):
# Token 0 kann sehen: [0]           ← nur sich selbst
# Token 1 kann sehen: [0, 1]        ← Vergangenheit + sich selbst
# Token 2 kann sehen: [0, 1, 2]     ← Vergangenheit + sich selbst
# Token 3 kann sehen: [0, 1, 2, 3]
# Token 4 kann sehen: [0, 1, 2, 3, 4]

seq_len = 5
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# Maske vor Softmax anwenden
scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
scores = scores.masked_fill(causal_mask, float('-inf'))  # -inf wird nach Softmax zu 0
attn_weights = torch.softmax(scores, dim=-1)

print("Kausale Aufmerksamkeitsgewichte (Token 2 kann nur Tokens 0,1,2 sehen):")
print(attn_weights[0, 2])  # Positionen 3 und 4 sind Null
print("\nJetzt lernt das Modell, 'Matte' vorherzusagen, ohne 'Matte' zuerst zu sehen!")

In [ ]:
# Schritt 3: Produktionsabkürzung (Eine Zeile)

# Du hast gerade den 4-Schritte-manuellen Prozess gelernt, um zu verstehen, was passiert.
# In der Praxis macht PyTorch alles für dich:

output = F.scaled_dot_product_attention(
    Q, K, V,
    is_causal=True  # wendet automatisch kausale Maskierung an
)

print(f"Ausgabeform: {output.shape}")  # (2, 5, 64)

# Warum dies statt manuell verwenden?
# - 2-4× schneller (nutzt FlashAttention)
# - Weniger Speicher (speichert nicht die volle Aufmerksamkeitsmatrix)
# - Behandelt Randfälle (numerische Stabilität, Dropout, Masken-Broadcasting)

print("\n✅ Wann manuell vs. Produktion verwenden:")
print("   Lernen: Manuell schreiben, um die Mathematik zu verstehen")
print("   Produktion: F.scaled_dot_product_attention immer verwenden")

In [ ]:
seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
print(mask)

## Broadcasting-Beispiele
Bias-Addition und Maskierung-Broadcast.

In [ ]:
x = torch.randn(3, 4)
y = x + 5
batch = torch.randn(32, 10, 768)
bias = torch.randn(768)
result = batch + bias  # Bias wird gebroadcastet
scores = torch.randn(4, 8, 10, 10)
mask = torch.triu(torch.ones(1, 1, 10, 10), 1)
masked = scores + mask * -1e9
print('Ergebnisform:', result.shape)

## Wesentliche Operationen: Elementweise, Matmul, Reduktionen

**Elementweise Operationen** arbeiten positionsweise — wie das Addieren von zwei Tabellen Zelle für Zelle. Wenn `a` und `b` beide 3×4 Tensoren sind, dann addiert `a + b` `a[0,0]` zu `b[0,0]`, `a[0,1]` zu `b[0,1]`, und so weiter. Gleiche Form rein, gleiche Form raus.

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(3, 4)
add = a + b
mul = a * b
square = a ** 2
exp = torch.exp(a)
x = torch.randn(32, 10, 64)
W = torch.randn(64, 128)
y = x @ W
total = a.sum()
row_sums = a.sum(dim=1, keepdim=True)
col_means = a.mean(dim=0)
max_vals, max_idx = a.max(dim=1)
c = torch.cat([a, b], dim=0)
d = torch.stack([a, b], dim=0)
logits = torch.randn(3, 5)
probs = torch.softmax(logits, dim=-1)
print('y Form:', y.shape)
print('Probs Zeilensummen:', probs.sum(dim=-1))

## Autograd: Automatische Gradienten

**Was ist ein Gradient?** Die Ableitung — um wie viel ändert sich die Ausgabe, wenn sich die Eingabe ändert?

**Warum wichtig?** Das Training eines neuronalen Netzes bedeutet, Millionen von Parametern anzupassen. Gradienten sagen uns, in welche Richtung wir anpassen sollen. PyTorchs Autograd macht dies automatisch.

In [ ]:
# Einfacher Berechnungsgraph: x → y → z
x = torch.tensor([2.0, 3.0], requires_grad=True)  # Operationen auf x verfolgen
y = x ** 2        # y = [4.0, 9.0]
z = y.sum()       # z = 13.0

# Gradienten automatisch berechnen
z.backward()      # "wie ändert sich z, wenn ich x ändere?"
print('Gradienten:', x.grad)  # tensor([4., 6.])

# Was ist gerade passiert?
# z = (x²).sum() → dz/dx = 2x
# Bei x=[2, 3] sind Gradienten 2*[2, 3] = [4, 6]
# backward() hat dies berechnet, indem es den Graphen rückwärts durchlaufen hat!

print('\n✅ Manuelle Überprüfung: dz/dx = 2x')
print(f'   Bei x=[2, 3]: 2*x = {2 * x.detach()}')

# Wann Verfolgung stoppen (spart Speicher während der Inferenz):
print('\n--- Gradientenverfolgung stoppen ---')

# Option 1: Kontextmanager (für einen Codeblock)
with torch.no_grad():
    y_no_grad = x * 2  # keine Gradientenverfolgung
    print(f'Kein Gradient berechnet: {y_no_grad}')

# Option 2: Detach (für einen einzelnen Tensor)
detached = x.detach()  # neuer Tensor, keine Gradient-Historie
print(f'Abgelöster Tensor: {detached}')

## Zusammenfassung

Du hast jetzt alle wichtigen PyTorch-Operationen für die LLM-Entwicklung gesehen:

✅ **Tensor-Grundlagen** - Erstellung, Datentypen, Geräte
✅ **Reproduzierbarkeit** - Seeds für Debugging
✅ **Dimensionsaufbau** - 1D → 2D → 3D → 4D Progression
✅ **Umformen & Indizierung** - Navigation durch mehrdimensionale Daten
✅ **Aufmerksamkeitsmechanismus** - manuell + Produktionsmuster
✅ **Kausale Maskierung** - zukünftiges Token-Spicken verhindern
✅ **Einbettungen** - Token + Positionsrepräsentationen
✅ **Broadcasting** - automatische Formerweiterung
✅ **Mathematische Operationen** - elementweise, Matmul, Reduktionen
✅ **Autograd** - automatische Differentiation
✅ **Trainingsschleifen** - vorwärts, rückwärts, optimieren mit Gradientenclipping

**Nächste Schritte:** Kapitel 7 zeigt dir, wie du echte Textdaten für das Training vorbereitest!